# Kiểm toán Dữ liệu Nguồn Scania APS (Source Audit)

**Dự án:** Tối ưu chi phí bảo trì dự đoán bằng Machine Learning  
**Dataset:** APS Failure at Scania Trucks (UCI ID: 421, DOI: 10.24432/C51S51)  
**Mã nhiệm vụ:** SC-G2-A (Antigravity Data Audit)  
**Phạm vi:** Kiểm toán tính toàn vẹn (integrity), cấu trúc và chất lượng tệp Train chính thức 60.000 mẫu (`aps_failure_training_set.csv`).  
**Nguyên tắc bảo vệ dữ liệu:** Tập Test 16.000 mẫu được niêm phong (sealed) bên trong tệp nén gốc, tuyệt đối không trích xuất, mở hoặc phân tích trước G10.

In [1]:
import os
import hashlib
import csv
import json
from collections import Counter, defaultdict

# Đường dẫn tệp dữ liệu Train
RAW_DIR = os.path.abspath(os.path.join("..", "..", "data", "raw", "scania_aps"))
TRAIN_CSV = os.path.join(RAW_DIR, "aps_failure_training_set.csv")
print(f"Tệp Train: {TRAIN_CSV}")
assert os.path.exists(TRAIN_CSV), f"Không tìm thấy tệp {TRAIN_CSV}"

Tệp Train: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\data\raw\scania_aps\aps_failure_training_set.csv


## 1. Kiểm tra SHA-256 Raw Bytes và Provenance

Xác thực mã băm SHA-256 và kích thước byte tệp dữ liệu nguồn Train.

In [2]:
sha256_hash = hashlib.sha256()
with open(TRAIN_CSV, "rb") as f:
    while chunk := f.read(65536):
        sha256_hash.update(chunk)
train_sha256 = sha256_hash.hexdigest()
file_size = os.path.getsize(TRAIN_CSV)

print(f"Kích thước file: {file_size:,} bytes")
print(f"SHA-256: {train_sha256}")
assert train_sha256 == "bb484302e3a3a1c8ef5e1f0129c4dc7cbd58f350867f95b575461ca21ab6b9da"

Kích thước file: 44,669,194 bytes
SHA-256: bb484302e3a3a1c8ef5e1f0129c4dc7cbd58f350867f95b575461ca21ab6b9da


## 2. Kiểm toán Preamble và Phát hiện Header

Tìm dòng header bắt đầu bằng cột `class` theo cấu trúc thực tế của tệp mà không hard-code số dòng bỏ qua.

In [3]:
preamble_lines = []
header_line_idx = None
header = None

with open(TRAIN_CSV, "r", encoding="utf-8", errors="replace") as f:
    for idx, line in enumerate(f):
        parts = [p.strip() for p in line.strip().split(",")]
        if len(parts) > 1 and parts[0] == "class":
            header_line_idx = idx
            header = parts
            break
        else:
            preamble_lines.append(line.strip())

print(f"Số dòng preamble/chú thích: {len(preamble_lines)} dòng")
print(f"Header xuất hiện ở dòng thứ: {header_line_idx + 1} (0-based index {header_line_idx})")
print(f"Tổng số cột: {len(header)}")
print(f"5 cột đầu tiên: {header[:5]}")
print(f"5 cột cuối cùng: {header[-5:]}")
assert header[0] == "class"
assert len(header) == 171

Số dòng preamble/chú thích: 20 dòng
Header xuất hiện ở dòng thứ: 21 (0-based index 20)
Tổng số cột: 171
5 cột đầu tiên: ['class', 'aa_000', 'ab_000', 'ac_000', 'ad_000']
5 cột cuối cùng: ['ee_007', 'ee_008', 'ee_009', 'ef_000', 'eg_000']


## 3. Kiểm toán Dữ liệu Dòng, Target và Missing Tokens

Kiểm tra tính toàn vẹn của 60.000 dòng dữ liệu, nhãn phân lớp và các token khuyết thiếu.

In [4]:
feature_names = header[1:]
row_count = 0
target_counts = Counter()
missing_counts = [0] * len(feature_names)
observed_counts = [0] * len(feature_names)
exact_x_dict = defaultdict(list)

with open(TRAIN_CSV, "r", encoding="utf-8", errors="replace") as f:
    for _ in range(header_line_idx + 1):
        next(f)
    
    reader = csv.reader(f)
    for row in reader:
        row_count += 1
        assert len(row) == 171, f"Dòng {row_count} có {len(row)} cột!"
        target = row[0].strip()
        target_counts[target] += 1
        
        canonical_x = []
        for col_idx, raw_val in enumerate(row[1:]):
            val_str = raw_val.strip()
            if val_str == "na":
                missing_counts[col_idx] += 1
                canonical_x.append(None)
            else:
                observed_counts[col_idx] += 1
                val_float = float(val_str)
                canonical_x.append(val_float)
        
        exact_x_dict[tuple(canonical_x)].append((row_count, target))

print(f"Tổng số dòng dữ liệu: {row_count:,}")
print(f"Phân bố nhãn: {dict(target_counts)}")
print(f"Tỷ lệ nhãn pos: {target_counts['pos'] / row_count:.4%}")
assert row_count == 60000
assert target_counts["neg"] == 59000
assert target_counts["pos"] == 1000

Tổng số dòng dữ liệu: 60,000
Phân bố nhãn: {'neg': 59000, 'pos': 1000}
Tỷ lệ nhãn pos: 1.6667%


## 4. Kiểm toán Exact-X Duplicates & Label Conflicts (Phê duyệt P03)

Kiểm tra xem có bất kỳ bản ghi nào trùng lặp hoàn toàn về thuộc tính (exact-X matching trước imputation) hoặc xung đột nhãn hay không.

In [5]:
unique_patterns = len(exact_x_dict)
duplicate_groups = sum(1 for rows in exact_x_dict.values() if len(rows) > 1)
conflicting_groups = sum(1 for rows in exact_x_dict.values() if len(set(t for _, t in rows)) > 1)

print(f"Tổng số mẫu đặc trưng duy nhất: {unique_patterns:,} / {row_count:,}")
print(f"Số nhóm trùng lặp (duplicate groups): {duplicate_groups}")
print(f"Số nhóm xung đột nhãn (conflicting label groups): {conflicting_groups}")

if duplicate_groups == 0 and conflicting_groups == 0:
    print("KẾT QUẢ KIỂM TOÁN P03: PASSED (Không có bản ghi trùng lặp exact-X hoặc xung đột nhãn)")
else:
    print("KẾT QUẢ KIỂM TOÁN P03: BLOCKED (Cần dừng lại để xin quyết định từ Chủ dự án)")
assert duplicate_groups == 0

Tổng số mẫu đặc trưng duy nhất: 60,000 / 60,000
Số nhóm trùng lặp (duplicate groups): 0
Số nhóm xung đột nhãn (conflicting label groups): 0
KẾT QUẢ KIỂM TOÁN P03: PASSED (Không có bản ghi trùng lặp exact-X hoặc xung đột nhãn)


## 5. Phân tích Chi tiết Khuyết thiếu (Missingness Profiling)

Thống kê các thuộc tính có tỷ lệ khuyết thiếu cao và nhận diện thuộc tính hằng số.

In [6]:
missing_rates = [c / row_count for c in missing_counts]
gt_50 = [(fn, missing_counts[i], missing_rates[i]) for i, fn in enumerate(feature_names) if missing_rates[i] > 0.50]
gt_50.sort(key=lambda x: x[2], reverse=True)

print(f"Tổng số thuộc tính có tỷ lệ missing > 50%: {len(gt_50)} thuộc tính")
for fn, cnt, rate in gt_50:
    print(f"  - {fn}: {cnt:,} / {row_count:,} ({rate:.2%})")

Tổng số thuộc tính có tỷ lệ missing > 50%: 8 thuộc tính
  - br_000: 49,264 / 60,000 (82.11%)
  - bq_000: 48,722 / 60,000 (81.20%)
  - bp_000: 47,740 / 60,000 (79.57%)
  - bo_000: 46,333 / 60,000 (77.22%)
  - ab_000: 46,329 / 60,000 (77.22%)
  - cr_000: 46,329 / 60,000 (77.22%)
  - bn_000: 44,009 / 60,000 (73.35%)
  - bm_000: 39,549 / 60,000 (65.92%)


## 6. Xác minh 7 Nhóm Đặc trưng Histogram

Xác nhận 7 nhóm đặc trưng dạng histogram (mỗi nhóm gồm 10 bins từ 000 đến 009, tổng cộng 70 đặc trưng).

In [7]:
hist_prefixes = ['ag', 'ay', 'az', 'ba', 'cn', 'cs', 'ee']
for pfx in hist_prefixes:
    bins = [fn for fn in feature_names if fn.startswith(f"{pfx}_")]
    print(f"Họ histogram '{pfx}': {len(bins)} bins ({bins[0]} đến {bins[-1]})")
    assert len(bins) == 10
print("Xác thực 7 họ đặc trưng histogram: 100% HOÀN TẤT (70 đặc trưng histogram + 100 đặc trưng số học đơn lẻ = 170 đặc trưng).")

Họ histogram 'ag': 10 bins (ag_000 đến ag_009)
Họ histogram 'ay': 10 bins (ay_000 đến ay_009)
Họ histogram 'az': 10 bins (az_000 đến az_009)
Họ histogram 'ba': 10 bins (ba_000 đến ba_009)
Họ histogram 'cn': 10 bins (cn_000 đến cn_009)
Họ histogram 'cs': 10 bins (cs_000 đến cs_009)
Họ histogram 'ee': 10 bins (ee_000 đến ee_009)
Xác thực 7 họ đặc trưng histogram: 100% HOÀN TẤT (70 đặc trưng histogram + 100 đặc trưng số học đơn lẻ = 170 đặc trưng).
